# 🔮 03. Fill Level Forecasting & Demand Prediction
### PreZero / Fillinq Challenge: Predictive Dispatch Intelligence & Chronological Validation

This notebook forecasts future container fill levels, volume ($m^3$), and weight ($kg$) based on a physics-grounded accumulation model:
$$\widehat{\text{Fill}}_i(\Delta t) = \min(110\%, \text{residual\_fill\_perc} + v_i \times \Delta t \times S_{\text{holiday}})$$

#### Crucial Operational Reality:
Every historical record in the PreZero dataset represents an **emptying visit** that reset the container back to 0% (or residual waste level). Therefore, forecasting fill level at future time $t + \Delta t$ should **NOT** add new growth onto an already-recorded pre-collection fill level! Instead, the container accumulates from 0% over elapsed time $\Delta t$ since its last emptying visit.

#### Key Enhancements in this Notebook:
1. **Physical Reset Modeling**: Growth accumulates from `residual_fill_perc` (default 0.0%) using the container's true daily fill velocity $v_i$.
2. **Chronological Model Validation**: Evaluated on an unseen chronological holdout test set (first 80% train, final 20% test), computing MAE and RMSE with +33.1% error reduction over baseline.
3. **Dynamic Lookahead Dispatch Forecasting**: Predicts demands across Heilbronn containers and classifies urgency into:
   - **Urgent** ($\ge 80\%$): Must be serviced to prevent overflows.
   - **Opportunistic** ($60\% - 79\%$): Efficiently collected if along the route.
   - **Safe** ($< 60\%$): No collection needed.
4. **Optimization Pipeline Compatibility**: Outputs `data/predicted_demands.csv` with all required columns for Gurobi CVRP route optimization.


In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.predictor import predict_container_demands, validate_fill_predictor

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['figure.dpi'] = 110

print("Predictor module ready.")

Predictor module ready.


---
## 📥 1. Load Clustered Container Profiles
Load profiles generated by `02_container_clustering.ipynb`.

In [2]:
cluster_path = os.path.join("data", "container_clusters.csv")
if os.path.exists(cluster_path):
    profiles = pd.read_csv(cluster_path)
else:
    from src.data_loader import load_and_clean_data
    from src.clustering import extract_container_profiles, cluster_containers
    df = load_and_clean_data()
    p = extract_container_profiles(df, material_filter='white')
    profiles, _ = cluster_containers(p)

print(f"Loaded {len(profiles)} container profiles.")
display(profiles[['Location_id', 'tier', 'avg_fill_perc', 'daily_fill_velocity', 'last_reading_date', 'current_fill_perc']].head(5))

Loaded 448 container profiles.
                            Location_id                                                tier  avg_fill_perc  daily_fill_velocity last_reading_date  current_fill_perc
0  0004c095-29ec-4894-81a8-05f3a7a55650                            Tier 3: Moderate Regular      80.000000             3.235294        2026-09-10               20.0
1  0058ed93-88f5-4c25-8746-2e5ee87e5559                            Tier 3: Moderate Regular      76.875000             3.600000        2026-09-17               80.0
2  009c98f3-c93a-4d55-abfe-79f7acf2a0ff                          Tier 2: Steady High-Demand      84.285714             2.149860        2026-08-25               80.0
3  00e9e2cf-b6f7-4872-908b-fcf4c35f388a  Tier 1: Critical High-Volume / Chronic Overfillers      94.705882             5.311355        2026-09-10              100.0
4  01371af2-75b4-4b9a-9e74-be20129483d5                       Tier 4: Low-Demand / Sporadic      58.000000             1.516517        2026-09-2

---
## 🧪 2. Rigorous Chronological Model Validation (80/20 Holdout)
To validate prediction accuracy without data leakage, we evaluate on an unseen future holdout test set:
* **Train Set**: First 80% chronologically (2025-10-24 to 2026-07-28, 3,169 collection events).
* **Holdout Test Set**: Final 20% chronologically (2026-07-29 to 2026-09-30, 812 collection events).
* We compute **Mean Absolute Error (MAE)** and **Root Mean Squared Error (RMSE)**, comparing against the naive baseline.

In [3]:
clean_path = os.path.join("data", "cleaned_fill_levels.csv")
if os.path.exists(clean_path):
    full_df = pd.read_csv(clean_path)
else:
    from src.data_loader import load_and_clean_data
    full_df = load_and_clean_data()

metrics, test_res = validate_fill_predictor(
    full_df,
    material_filter='white',
    train_ratio=0.8,
    residual_fill_perc=0.0,
    holiday_multiplier=1.25
)

val_summary = pd.DataFrame({
    'Metric / Property': [
        'Training Date Range',
        'Holdout Test Date Range',
        'Training Sample Count',
        'Holdout Test Sample Count',
        'Model MAE (% points)',
        'Baseline MAE (% points)',
        'MAE Error Reduction (%)',
        'Model RMSE (% points)',
        'Baseline RMSE (% points)',
        'Actual Test Mean Fill (%)',
        'Predicted Test Mean Fill (%)'
    ],
    'Value': [
        metrics['train_date_range'],
        metrics['test_date_range'],
        f"{metrics['train_samples']:,}",
        f"{metrics['test_samples']:,}",
        f"{metrics['mae']:.2f}%",
        f"{metrics['baseline_mae']:.2f}%",
        f"+{metrics['mae_improvement_pct']:.1f}%",
        f"{metrics['rmse']:.2f}%",
        f"{metrics['baseline_rmse']:.2f}%",
        f"{metrics['actual_test_mean']:.2f}%",
        f"{metrics['predicted_test_mean']:.2f}%"
    ]
})

print("=" * 75)
print("             CHRONOLOGICAL HOLDOUT MODEL VALIDATION RESULTS")
print("=" * 75)
display(val_summary)

             CHRONOLOGICAL HOLDOUT MODEL VALIDATION RESULTS
               Metric / Property                     Value
0            Training Date Range  2025-10-24 to 2026-07-28
1        Holdout Test Date Range  2026-07-29 to 2026-09-30
2          Training Sample Count                     3,169
3      Holdout Test Sample Count                       812
4           Model MAE (% points)                    17.81%
5        Baseline MAE (% points)                    26.63%
6        MAE Error Reduction (%)                    +33.1%
7          Model RMSE (% points)                    24.86%
8       Baseline RMSE (% points)                    34.06%
9      Actual Test Mean Fill (%)                    79.98%
10  Predicted Test Mean Fill (%)                    80.75%


### 📊 Model Calibration & Error Distribution on Unseen Holdout Data

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Calibration Plot: Actual vs Predicted
axes[0].scatter(test_res['pred_fill'], test_res['fill_num'], alpha=0.35, color='#1f77b4', edgecolors='none', s=35)
axes[0].plot([0, 110], [0, 110], color='red', linestyle='--', linewidth=2, label='Perfect Calibration (y = x)')
axes[0].set_title(f"Model Calibration on Holdout Test Set (N={len(test_res)})", fontweight='bold')
axes[0].set_xlabel("Predicted Fill Level (%)")
axes[0].set_ylabel("Actual Observed Fill Level (%)")
axes[0].set_xlim(0, 115)
axes[0].set_ylim(0, 115)
axes[0].legend(fontsize=10)

# 2. Prediction Error Distribution
test_res['error'] = test_res['pred_fill'] - test_res['fill_num']
sns.histplot(test_res['error'], kde=True, color='#2ca02c', ax=axes[1], bins=30)
axes[1].axvline(0, color='black', linestyle='--', alpha=0.7)
axes[1].axvline(metrics['mae'], color='red', linestyle=':', label=f"MAE: {metrics['mae']:.1f}%")
axes[1].axvline(-metrics['mae'], color='red', linestyle=':')
axes[1].set_title("Forecast Error Distribution (Predicted - Actual)", fontweight='bold')
axes[1].set_xlabel("Prediction Error (% points)")
axes[1].set_ylabel("Frequency")
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## 🔮 3. Dynamic Lookahead Dispatch Forecasting
Simulate dispatch planning for a **2-day lookahead** with holiday season surge:
* **Lookahead ($\Delta t$):** Days elapsed since container's last collection visit + 2 planning days.
* **Holiday Multiplier ($S_{\text{holiday}}$):** $1.20$ (+20% consumption surge).
* **Container Volume:** $2.5\text{ m}^3$
* **Glass Bulk Density:** $380\text{ kg/m}^3$
* **Residual Fill Level:** $0.0\%$ (standard complete gravity discharge)

In [5]:
# Forecast container fill levels and physical demands
demands = predict_container_demands(
    profiles_df=profiles,
    days_ahead=2,
    holiday_multiplier=1.20,
    residual_fill_perc=0.0,
    container_volume_m3=2.5,
    glass_density_kg_m3=380.0
)

urgency_dist = demands['urgency_status'].value_counts()
print("=" * 65)
print("          DISPATCH URGENCY CLASSIFICATION (2-DAY FORECAST)")
print("=" * 65)
display(pd.DataFrame({
    'Container Count': urgency_dist,
    'Fleet Share (%)': (urgency_dist / len(demands) * 100).round(1)
}))

# Tier breakdown by urgency
tier_urgency = pd.crosstab(demands['tier'], demands['urgency_status'], margins=True)
print("\nUrgency Distribution Across Behavioral Tiers:")
display(tier_urgency)

          DISPATCH URGENCY CLASSIFICATION (2-DAY FORECAST)
                        Container Count  Fleet Share (%)
urgency_status                                          
Urgent (>=80%)                      198             44.2
Safe (<60%)                         179             40.0
Opportunistic (60-79%)               71             15.8

Urgency Distribution Across Behavioral Tiers:
urgency_status                                      Opportunistic (60-79%)  Safe (<60%)  Urgent (>=80%)  All
tier                                                                                                        
Tier 1: Critical High-Volume / Chronic Overfillers                      13           33              70  116
Tier 2: Steady High-Demand                                              18           54              55  127
Tier 3: Moderate Regular                                                22           47              55  124
Tier 4: Low-Demand / Sporadic                                   

---
## 📊 4. Urgency and Demand Distribution Visualizations

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

status_colors = {
    'Urgent (>=80%)': '#d62728',
    'Opportunistic (60-79%)': '#ff7f0e',
    'Safe (<60%)': '#2ca02c'
}

# 1. Container Count by Urgency Status
sns.countplot(data=demands, x='urgency_status', palette=status_colors, ax=axes[0])
axes[0].set_title("Container Count by Urgency Status", fontweight='bold')
axes[0].set_xlabel("Urgency Status")
axes[0].set_ylabel("Number of Containers")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

# 2. Demand Weight Distribution for Urgent Bins
urgent_demands = demands[demands['urgency_status'] == 'Urgent (>=80%)']
sns.histplot(urgent_demands['demand_weight_kg'], kde=True, color='#d62728', ax=axes[1], bins=20)
axes[1].set_title(f"Payload Demand Distribution (kg) for Urgent Bins (N={len(urgent_demands)})", fontweight='bold')
axes[1].set_xlabel("Estimated Weight (kg)")
axes[1].set_ylabel("Container Count")

# 3. Predicted Fill Level by Operational Tier
tier_order = [
    'Tier 1: Critical High-Volume / Chronic Overfillers',
    'Tier 2: Steady High-Demand',
    'Tier 3: Moderate Regular',
    'Tier 4: Low-Demand / Sporadic'
]
sns.boxplot(data=demands, x='tier', y='predicted_fill_perc', order=tier_order, palette='Set2', ax=axes[2])
axes[2].set_title("Predicted Fill Level (%) by Behavioral Tier", fontweight='bold')
axes[2].set_xlabel("Behavioral Tier")
axes[2].set_ylabel("Predicted Fill (%)")
axes[2].axhline(80, color='red', linestyle='--', alpha=0.7, label='Urgent Threshold (80%)')
axes[2].set_xticklabels(['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4'])
axes[2].legend(loc='lower right')

plt.tight_layout()
plt.show()

print(f"Total Urgent Collection Demand: {urgent_demands['demand_volume_m3'].sum():.1f} m³ | {urgent_demands['demand_weight_kg'].sum() / 1000.0:.2f} metric tons")
print(f"Total Opportunistic Volume: {demands[demands['urgency_status'] == 'Opportunistic (60-79%)']['demand_volume_m3'].sum():.1f} m³")

Total Urgent Collection Demand: 506.4 m³ | 192.44 metric tons
Total Opportunistic Volume: 122.9 m³


---
## 💾 5. Save Predicted Demands for Optimization
Preserve all downstream columns required by `04_gurobi_route_optimization.ipynb`:
`['Location_id', 'tier', 'predicted_fill_perc', 'demand_volume_m3', 'demand_weight_kg', 'urgency_status']`.

In [7]:
output_path = os.path.join("data", "predicted_demands.csv")
demands.to_csv(output_path, index=False)
print(f"Predicted demands saved to: {output_path}")

# Verify backward compatibility
required_cols = ['Location_id', 'tier', 'predicted_fill_perc', 'demand_volume_m3', 'demand_weight_kg', 'urgency_status']
missing = [c for c in required_cols if c not in demands.columns]
if not missing:
    print(f"✓ All required downstream columns verified: {required_cols}")
else:
    raise AssertionError(f"Missing required columns: {missing}")

print(f"Total candidate urgent containers for routing: {len(urgent_demands)}")

Predicted demands saved to: data\predicted_demands.csv
✓ All required downstream columns verified: ['Location_id', 'tier', 'predicted_fill_perc', 'demand_volume_m3', 'demand_weight_kg', 'urgency_status']
Total candidate urgent containers for routing: 198
